# CPU Baseline

**Runtime:** Google Colab, CPU runtime

This notebook produces the CPU reference timings for our benchmark. It is one of three
notebooks:

1. **`CPU_Baseline.ipynb`** (this one): runs the pipeline on CPU and writes
   `results/timings_cpu.csv` plus `results/best_params.json`.
2. **`GPU_Solution.ipynb`**: runs the same pipeline on GPU using cuDF / cuML / XGBoost-GPU,
   reuses the hyperparameters from this notebook, and writes `results/timings_gpu.csv`.
3. **`Analysis.ipynb`**: joins the two CSVs and produces the speedup tables and figures
   used in the report.

Both timing notebooks are deliberately structured as mirrors of each other. Only the
execution engine changes, so any timing difference is attributable to hardware, not to
modelling choices.

## The underlying ML task

The dataset is the CFPB Consumer Complaints corpus, reused from a previous ML assignment
(`72782_Complaints_Notebook.ipynb`). The target is binary: did the consumer dispute the
company's response (`Consumer disputed?`, Yes/No)? We keep the original feature engineering
(target-encoded categoricals, narrative-derived text features, temporal flags,
response-quality flags) and the original two-model setup:

1. **Logistic Regression** (L2-regularised, class-balanced): the linear baseline.
2. **XGBoost** (histogram tree booster): the non-linear competitor.

What is new here is that we re-instrument the pipeline to measure wall-clock time at each
stage.

## How the notebook is organised

The notebook runs in two functionally distinct phases:

**Phase A, Hyperparameter search (Section 4).** A single, full-data CV search per model
(`GridSearchCV` for LR, `RandomizedSearchCV` for XGBoost). Best parameters are persisted
to `results/best_params.json` so the sweep in Section 5 (and the GPU notebook) all time
the *same* model. The CV calls themselves are also timed and serve as a context baseline.

**Phase B, Dataset-size sweep (Section 5).** The actual benchmark. For each of 5 fractions
(10%, 25%, 50%, 75%, 100% of the training set), we re-run the full pipeline and time each
phase separately. Each (fraction, phase) combination is repeated 3 times so the analysis
notebook can take the median and shake off scheduler noise. That gives
**3 runs × 5 fractions × 6 phases = 90 timing rows** in `results/timings_cpu.csv`.
A separate HPO sub-sweep (5.3) times `lr_cv` and `xgb_cv` at three fractions and appends
to the same CSV.

### What each phase measures

| Phase         | Stage         | What runs in this timed block                                                            |
|---------------|---------------|------------------------------------------------------------------------------------------|
| `fe_train`    | Feature eng.  | Fit + transform feature engineering on the training fold (target encoders learn from y) |
| `fe_test`     | Feature eng.  | Apply the fitted encoders to the test set (transform-only, no learning)                  |
| `lr_fit`      | Training      | Fit Logistic Regression on the engineered training matrix                                |
| `xgb_fit`     | Training      | Fit XGBoost on the same engineered training matrix                                       |
| `lr_predict`  | Inference     | `predict_proba` of LR on the engineered test matrix                                      |
| `xgb_predict` | Inference     | `predict_proba` of XGBoost on the engineered test matrix                                 |
| `lr_cv`       | HPO (5.3)     | Full 5-fold `GridSearchCV` for LR, one run per fraction                                  |
| `xgb_cv`      | HPO (5.3)     | Full 5-fold `RandomizedSearchCV` for XGBoost, one run per fraction                       |

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Train/Test Split](#2-data-loading--traintest-split)
3. [Feature Engineering](#3-feature-engineering)
4. [Hyperparameter Search (Context Timing)](#4-hyperparameter-search-context-timing)
    - 4.1 [Logistic Regression — GridSearchCV](#41-logistic-regression--gridsearchcv)
    - 4.2 [XGBoost — RandomizedSearchCV](#42-xgboost--randomizedsearchcv)
    - 4.3 [Persist Best Parameters](#43-persist-best-parameters)
5. [Dataset-Size Sweep](#5-dataset-size-sweep)
    - 5.1 [Timing Utility](#51-timing-utility)
    - 5.2 [Fit/Predict Sweep](#52-fitpredict-sweep)
    - 5.3 [HPO Sweep](#53-hpo-sweep)
    - 5.4 [Sanity Check](#54-sanity-check)

## 1. Setup & Imports

In [ ]:
# importing necessary libraries
import numpy as np
import pandas as pd
import time
import json
import os
import platform
import sys
import subprocess
import sklearn
import xgboost

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Fix random seed for reproducibility
np.random.seed(42)

In [ ]:
# Environment fingerprint to pin the report's performance numbers to a specific hardware/software stack so the CPU vs GPU comparison is reproducible and the figures can be cited precisely

print('--- Software ---')
print(f'Python         : {sys.version.split()[0]}')
print(f'NumPy          : {np.__version__}')
print(f'Pandas         : {pd.__version__}')
print(f'scikit-learn   : {sklearn.__version__}')
print(f'XGBoost        : {xgboost.__version__}')

print('\n--- Hardware ---')
print(f'Platform       : {platform.platform()}')
print(f'Processor      : {platform.processor() or "n/a"}')
try:
    cpu = subprocess.check_output(['lscpu'], text=True)
    for line in cpu.splitlines():
        if any(k in line for k in ('Model name', 'CPU(s):', 'Thread(s) per core',
                                   'Core(s) per socket', 'CPU max MHz', 'L3 cache')):
            print(line.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print('lscpu not available (non-Linux host)')

try:
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith(('MemTotal', 'MemAvailable')):
                print(line.strip())
except FileNotFoundError:
    pass

--- Software ---
Python         : 3.12.13
NumPy          : 2.0.2
Pandas         : 2.2.2
scikit-learn   : 1.6.1
XGBoost        : 3.2.0

--- Hardware ---
Platform       : Linux-6.6.113+-x86_64-with-glibc2.35
Processor      : x86_64
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
L3 cache:                                55 MiB (1 instance)
NUMA node0 CPU(s):                       0,1
MemTotal:       13286948 kB
MemAvailable:   12110232 kB


In [ ]:
# Persist outputs to Drive so they survive VM reclamation
# To re-run from scratch: delete /content/drive/MyDrive/results/timings_cpu.csv first,
# otherwise Section 5 appends to the existing 90 rows and the sanity check in 5.4 fails

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.chdir('/content')
    if not os.path.exists('results'):
        os.symlink(RESULTS_DIR, 'results')
    print(f'results/ now points to {RESULTS_DIR}')
except ImportError:
    # Local fallback (non-Colab): keep the existing behaviour
    os.makedirs('results', exist_ok=True)
    print('Local mode: results/ is a regular directory')

Mounted at /content/drive
results/ now points to /content/drive/MyDrive/results


## 2. Data Loading & Train/Test Split

We use the same CFPB dataset and the same 80/20 stratified split
(`random_state=42`) as the original ML notebook. The target (`Consumer disputed?`)
is heavily imbalanced (~21% positive), so the split is stratified
on `y` to preserve the class ratio in both sets.

In [4]:
# Load the training CSV
# Default: Google Drive (Colab). Falls back to local path if Drive isn't available.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/complaints_training.csv'
except ImportError:
    DATA_PATH = 'complaints_training.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df_raw):,} rows × {df_raw.shape[1]} columns from {DATA_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 321,430 rows × 18 columns from /content/drive/MyDrive/complaints_training.csv


In [5]:
# Features: everything except the ID column and the target itself.
# Target: binary-encode 'Consumer disputed?' to {0, 1}.
X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

# 80/20 stratified split
# Seed is the same as in the original ML notebook resulting in same splits
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train: {len(X_train_raw):,} rows  |  positive rate: {y_train.mean():.3f}')
print(f'Test:  {len(X_test_raw):,} rows  |  positive rate: {y_test.mean():.3f}')

Train: 257,144 rows  |  positive rate: 0.199
Test:  64,286 rows  |  positive rate: 0.199


## 3. Feature Engineering

All feature construction was consolidated into a single `feature_engineering()`
function.

**Output features (~41 columns) fall into five groups:**

- **Temporal** — response lag (days between receipt and forwarding), received
  year, day of week.
- **Response-quality flags** — regex-derived binaries on the company response
  text (monetary relief, explanation only, etc.), timely-response flag,
  interactions.
- **Narrative-derived** — word count, char count, average word length, presence
  of legal/escalation terms (`attorney`, `lawsuit`, `fraud`, …).
- **Metadata flags** — consent provided/withdrawn, sub-product/sub-issue
  present, vulnerability tags (older Americans, servicemembers).
- **Encoded categoricals** — smoothed target encoding (with additive smoothing,
  α=20) for high-cardinality columns (`Product`, `Company`, `State`, `ZIP3`,
  interactions), plus frequency encoding for `Company`.

**Train vs. inference mode.** The function has two modes controlled by
`fit_encoders`:
- `fit_encoders=None` — **training mode**: encoders are fit on the incoming
  data (computes per-category target means etc.) and the transformed frame is
  returned alongside the fitted encoders.
- `fit_encoders=<dict>` — **inference mode**: pre-fitted encoders are applied
  as-is, with no re-fitting. This prevents test-set leakage.

In the sweep below, `fe_train` calls the function in training mode on the
fraction-sized training subset; `fe_test` applies the resulting encoders to
the full (fraction-independent) test set.

In [6]:
def feature_engineering(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None
    raw = X.copy()
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])
    out = pd.DataFrame(index=raw.index)

    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')
    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',    case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief', case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',    case=False).astype(int)
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative']       = (narr != '').astype(int)
    out['word_count']          = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']          = narr.str.len().fillna(0).astype(int)
    out['avg_word_length']     = narr.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    out['narrative_short']     = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']      = (out['word_count'] > 200).astype(int)
    esc = 'attorney|lawyer|lawsuit|legal action|sue|court|cfpb|regulation|violation|fraud|illegal|discriminat'
    out['escalation_term_count'] = narr.str.lower().str.count(esc).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)

    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values}, index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode(
        raw['ZIP code'].fillna('000').astype(str).str[:3],                                             'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out, encoders


The cell below runs feature engineering once on the full training set. Its
output is consumed only by the hyperparameter search in section 4 so the CV
sees the full 257k-row training set when picking best parameters. The
dataset-size sweep in section 5 re-runs feature engineering from scratch on
each fraction, so nothing here is reused there.

In [ ]:
# Fit encoders on the full training set; apply to the test set with no refitting.
# Used only by the hyperparameter search in section 4.
# The dataset-size sweep in section 5 rebuilds features per fraction and does not reuse these objects.

X_train_enc, encoders = feature_engineering(X_train_raw, y=y_train)
X_test_enc, _         = feature_engineering(X_test_raw, fit_encoders=encoders)

assert X_train_enc.isnull().sum().sum() == 0, 'Nulls in training features — check FE'
assert X_test_enc.isnull().sum().sum()  == 0, 'Nulls in test features — check FE'
print(f'Features: {X_train_enc.shape[1]}  |  train rows: {len(X_train_enc):,}  |  test rows: {len(X_test_enc):,}')

Features: 41  |  train rows: 257,144  |  test rows: 64,286


## 4. Hyperparameter Search (Context Timing)

Both models are tuned with stratified 5-fold cross-validation on the full training set.
Best parameters are written to `results/best_params.json` and reused by the dataset-size
sweep in Section 5, so every fit there uses the same model. The CV calls themselves are
timed and re-run at multiple fractions in Section 5.3 as the `lr_cv` and `xgb_cv` phases.

### 4.1 Logistic Regression — GridSearchCV

**Model.** L2-regularised Logistic Regression with class-balanced weights, the linear
baseline against which XGBoost must prove its worth.

**Grid.** Five values of `C` spanning four orders of magnitude, `penalty='l2'` only.

**Scoring.** F1, since the target is imbalanced (~21% positive) and F1 was the metric
used in the original assignment.

**Solver.** `lbfgs`, sklearn's standard quasi-Newton solver for L2-regularised LR.

**`n_jobs=1`.** CV folds run serially. XGBoost (tuned next) already parallelises via
OpenMP, and stacking parallel CV on top produces noisy timings on the 2-core Colab free
tier. Keeping `n_jobs=1` for both searches makes their context timings directly comparable.

In [ ]:
# LR GridSearchCV — one-off, full training set.
# Produces best hyperparameters (saved to JSON) and a context timing

lr_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# L2-only grid: required for cross-framework equivalence with cuML's QN solver.
param_grid_lr = {
    'classifier__C':            [0.001, 0.01, 0.1, 1, 10],
    'classifier__penalty':      ['l2'],
    'classifier__class_weight': ['balanced'],
    'classifier__solver':       ['lbfgs'],
}

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_lr = GridSearchCV(
    lr_pipeline, param_grid_lr,
    cv=cv_lr, scoring='f1', n_jobs=1, verbose=1,
)

t0 = time.perf_counter()
search_lr.fit(X_train_enc, y_train)
t_lr_cv = time.perf_counter() - t0

# Score the best model on the held-out test set.
y_prob_lr = search_lr.best_estimator_.predict_proba(X_test_enc)[:, 1]
auc_lr    = roc_auc_score(y_test, y_prob_lr)

n_fits = len(search_lr.cv_results_['params']) * cv_lr.get_n_splits()
print(f'Best LR params: {search_lr.best_params_}')
print(f'CV F1:          {search_lr.best_score_:.4f}')
print(f'Test ROC-AUC:   {auc_lr:.4f}')
print(f'[CONTEXT] LR GridSearchCV ({n_fits} fits): {t_lr_cv:.1f}s  ({t_lr_cv/60:.1f} min)')

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best LR params: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
CV F1:          0.3915
Test ROC-AUC:   0.6422
[CONTEXT] LR GridSearchCV (25 fits): 138.1s  (2.3 min)


### 4.2 XGBoost — RandomizedSearchCV

**Model.** Gradient-boosted decision trees with histogram-based splitting. Non-linear,
captures feature interactions that LR cannot.

**Grid.** Nine hyperparameters covering tree complexity, regularisation, stochasticity,
imbalance handling, and the boosting budget. The full grid has ~46,000 combinations, so
exhaustive search is infeasible. We use `RandomizedSearchCV` with 60 samples × 5 folds =
**300 fits**.

**`tree_method='hist'`, `device='cpu'`.** Histogram tree building with the CPU backend.

**Scoring and `n_jobs`.** F1 and `n_jobs=1`, matching 4.1. XGBoost already saturates the
2 Colab cores via OpenMP, so parallelising CV on top would only add scheduler noise.

In [ ]:
# XGBoost RandomizedSearchCV: one-off, full training set.
# Produces best hyperparameters (saved to JSON) and a context timing.

xgb_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss',
        tree_method='hist', device='cpu',
    ))
])

param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7],
}

cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_xgb = RandomizedSearchCV(
    xgb_pipeline, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1,
)

t0 = time.perf_counter()
search_xgb.fit(X_train_enc, y_train)
t_xgb_cv = time.perf_counter() - t0

# Score the best model on the held-out test set.
y_prob_xgb = search_xgb.best_estimator_.predict_proba(X_test_enc)[:, 1]
auc_xgb    = roc_auc_score(y_test, y_prob_xgb)

n_fits = search_xgb.n_iter * cv_xgb.get_n_splits()
print(f'Best XGB params: {search_xgb.best_params_}')
print(f'CV F1:           {search_xgb.best_score_:.4f}')
print(f'Test ROC-AUC:    {auc_xgb:.4f}')
print(f'[CONTEXT] XGB RandomizedCV ({n_fits} fits): {t_xgb_cv:.1f}s  ({t_xgb_cv/60:.1f} min)')

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best XGB params: {'classifier__subsample': 0.65, 'classifier__scale_pos_weight': 4, 'classifier__reg_lambda': 5, 'classifier__reg_alpha': 0.1, 'classifier__n_estimators': 500, 'classifier__min_child_weight': 15, 'classifier__max_depth': 6, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 0.6}
CV F1:           0.3988
Test ROC-AUC:    0.6520
[CONTEXT] XGB RandomizedCV (300 fits): 6079.9s  (101.3 min)


### 4.3 Persist Best Parameters

The best hyperparameters, test-set AUCs, and context timings are written to
`results/best_params.json`.

In [ ]:
# Persist best hyperparameters, AUCs, and context timings for GPU_Solution.ipynb.
os.makedirs('results', exist_ok=True)

# Strip the sklearn Pipeline prefix so the dict can be unpacked into
# model constructors on the GPU side without renaming.
best_lr_params  = {k.replace('classifier__', ''): v for k, v in search_lr.best_params_.items()}
best_xgb_params = {k.replace('classifier__', ''): v for k, v in search_xgb.best_params_.items()}

best_params = {
    'lr':  best_lr_params,
    'xgb': best_xgb_params,
    'auc': {
        'lr':  float(auc_lr),
        'xgb': float(auc_xgb),
    },
    'cv_context_seconds': {
        'lr_gridsearch':        t_lr_cv,
        'xgb_randomizedsearch': t_xgb_cv,
    },
}

out_path = 'results/best_params.json'
with open(out_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'Saved {out_path}')
print(json.dumps(best_params, indent=2))

Saved results/best_params.json
{
  "lr": {
    "C": 0.1,
    "class_weight": "balanced",
    "penalty": "l2",
    "solver": "lbfgs"
  },
  "xgb": {
    "subsample": 0.65,
    "scale_pos_weight": 4,
    "reg_lambda": 5,
    "reg_alpha": 0.1,
    "n_estimators": 500,
    "min_child_weight": 15,
    "max_depth": 6,
    "learning_rate": 0.05,
    "colsample_bytree": 0.6
  },
  "auc": {
    "lr": 0.6421907129973952,
    "xgb": 0.6520096289775927
  },
  "cv_context_seconds": {
    "lr_gridsearch": 138.05648630599995,
    "xgb_randomizedsearch": 6079.864166348
  }
}


## 5. Dataset-Size Sweep

The main benchmark. For each of 5 fractions (10%, 25%, 50%, 75%, 100%), we re-run the full
pipeline and time six phases: `fe_train`, `fe_test`, `lr_fit`, `lr_predict`, `xgb_fit`,
`xgb_predict`. Each combination is repeated 3 times to estimate variance, giving
**3 × 5 × 6 = 90 rows** in `results/timings_cpu.csv`. Section 5.3 appends an HPO sweep
(`lr_cv`, `xgb_cv`) at a subset of fractions, which feeds the report's HPO speedup figure.

### 5.1 Timing Utility

All timed calls go through `log_row()`, which writes to `results/timings_cpu.csv` with a
fixed schema.

**Schema:** `device`, `fraction`, `n_rows`, `phase`, `run_idx`, `seconds`, `notes`.

In [ ]:
# Timing utility. Every timed call in the sweep below goes through log_row()
# to guarantee a consistent schema for the analysis notebook.

_rows = []

def log_row(device, fraction, n_rows, phase, run_idx, seconds, notes=''):
    """Append one timing record to the in-memory buffer.

    Columns: device, fraction, n_rows, phase, run_idx, seconds, notes.
    Buffer is flushed to CSV after every outer run of the sweep.
    """
    _rows.append({
        'device':   device,
        'fraction': fraction,
        'n_rows':   n_rows,
        'phase':    phase,
        'run_idx':  run_idx,
        'seconds':  seconds,
        'notes':    notes,
    })

### 5.2 Fit/Predict Sweep

Three outer runs × five fractions × six phases. Each iteration builds a fresh model from
the best hyperparameters loaded in section 4, runs feature engineering on the fraction's
training subset, fits and predicts with LR and XGBoost, and logs wall-clock time for every
phase.

**Design choices:**

- **Nested subsets.** All fractions use `random_state=42`, so the 10% subset is contained
  in the 25%, which is contained in the 50%, etc. This isolates dataset-size effects from
  sampling effects.
- **Three runs, same seed.** Repeated runs time the *same* subset to capture pure timing
  variance (CPU scheduling, cache state, memory pressure). Sampling variance is excluded
  by design.
- **Fresh model per iteration.** LR and XGB are reconstructed inside the loop, preventing
  warm-start or cached state from contaminating timings.
- **Fraction-specific `fe_test` encoders.** Test-set FE uses encoders fit on the
  fraction-sized training subset, not the full training set, so `fe_test` reflects real
  production behaviour and scales with encoder size.
- **Hyperparameters fixed.** Loaded from `best_params.json` and reused across all
  fractions. Re-tuning per fraction would conflate model-quality variance with timing
  variance; the cost of tuning itself is measured separately in 5.3.
- **CSV flushed after each outer run.** Colab sessions can die mid-sweep (idle timeout,
  preemption), so at least one complete run is always recoverable.

In [ ]:
# Dataset-size sweep (fit/predict phases).
# 3 runs × 5 fractions × 6 phases = 90 rows, written to results/timings_cpu.csv.

# Reset state so re-running this cell produces a clean CSV
# (otherwise rows from a previous run would be appended).
_rows = []
_csv_path = 'results/timings_cpu.csv'
if os.path.exists(_csv_path):
    os.remove(_csv_path)
    print(f'Removed existing {_csv_path}; starting fresh sweep.')

FRACS = [0.10, 0.25, 0.50, 0.75, 1.00]
RUNS  = 3

for run_idx in range(RUNS):
    run_start = time.perf_counter()
    for frac in FRACS:
        # Nested subsets: same seed across fractions means 10% ⊂ 25% ⊂ … ⊂ 100%.
        X_sub = X_train_raw.sample(frac=frac, random_state=42)
        y_sub = y_train.loc[X_sub.index]
        n = len(X_sub)

        # --- Feature engineering (train) -------------------------------
        t0 = time.perf_counter()
        X_sub_enc, enc_sub = feature_engineering(X_sub, y=y_sub)
        log_row('cpu', frac, n, 'fe_train', run_idx, time.perf_counter() - t0)

        # --- Feature engineering (test) --------------------------------
        # Uses fraction-specific encoders, reflects real production usage.
        t0 = time.perf_counter()
        X_test_sub, _ = feature_engineering(X_test_raw, fit_encoders=enc_sub)
        log_row('cpu', frac, n, 'fe_test', run_idx, time.perf_counter() - t0)

        # --- Logistic Regression ---------------------------------------
        # Fresh pipeline per iteration → no state leakage across runs.
        lr = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('scaler',     StandardScaler()),
            ('classifier', LogisticRegression(
                **best_lr_params, random_state=42, max_iter=1000
            )),
        ])
        t0 = time.perf_counter()
        lr.fit(X_sub_enc, y_sub)
        log_row('cpu', frac, n, 'lr_fit', run_idx, time.perf_counter() - t0)

        t0 = time.perf_counter()
        lr.predict_proba(X_test_sub)
        log_row('cpu', frac, n, 'lr_predict', run_idx, time.perf_counter() - t0)

        # --- XGBoost ---------------------------------------------------
        # Fresh pipeline per iteration to avoid state leakage across runs.
        xgb = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(
                **best_xgb_params,
                tree_method='hist', device='cpu',
                random_state=42, eval_metric='logloss',
            )),
        ])
        t0 = time.perf_counter()
        xgb.fit(X_sub_enc, y_sub)
        log_row('cpu', frac, n, 'xgb_fit', run_idx, time.perf_counter() - t0)

        t0 = time.perf_counter()
        xgb.predict_proba(X_test_sub)
        log_row('cpu', frac, n, 'xgb_predict', run_idx, time.perf_counter() - t0)

        print(f'run={run_idx} frac={frac:.0%} n={n:,} done')

    # Flush after each outer run so the run is recoverable if Colab dies mid-sweep.
    pd.DataFrame(_rows).to_csv('results/timings_cpu.csv', index=False)
    run_wall = time.perf_counter() - run_start
    print(f'[run {run_idx}] wall-clock: {run_wall/60:.1f} min  |  rows so far: {len(_rows)}  |  saved results/timings_cpu.csv')

run=0 frac=10% n=25,714 done
run=0 frac=25% n=64,286 done
run=0 frac=50% n=128,572 done
run=0 frac=75% n=192,858 done
run=0 frac=100% n=257,144 done
[run 0] wall-clock: 1.9 min  |  rows so far: 30  |  saved results/timings_cpu.csv
run=1 frac=10% n=25,714 done
run=1 frac=25% n=64,286 done
run=1 frac=50% n=128,572 done
run=1 frac=75% n=192,858 done
run=1 frac=100% n=257,144 done
[run 1] wall-clock: 2.0 min  |  rows so far: 60  |  saved results/timings_cpu.csv
run=2 frac=10% n=25,714 done
run=2 frac=25% n=64,286 done
run=2 frac=50% n=128,572 done
run=2 frac=75% n=192,858 done
run=2 frac=100% n=257,144 done
[run 2] wall-clock: 2.1 min  |  rows so far: 90  |  saved results/timings_cpu.csv


### 5.3 HPO Sweep

Cross-validated hyperparameter search is itself an embarrassingly parallel workload
(`n_iter × k_folds` independent fits) and a key scaling axis where parallel hardware can
pay off. This section re-runs the searches from 4.1 (LR) and 4.2 (XGB) at multiple
fractions and appends two new phases to the timing log:

- `lr_cv`: full L2 grid, 5 folds, 25 fits per fraction.
- `xgb_cv`: randomized search, 60 samples × 5 folds, 300 fits per fraction.

**Fractions measured.** Both phases run at 25%, 50%, and 100%, giving 3 × 2 = **6 HPO rows**.

**Single run per fraction.** We measure the *scaling curve*, not the variance. The 4.1
and 4.2 context timings already cover the 100% baseline.

In [ ]:
# HPO sweep: time the full hyperparameter search at multiple fractions.
# Adds the lr_cv / xgb_cv phases needed by Analysis.ipynb to compute
# HPO speedup at each fraction.

HPO_FRACS = [0.25, 0.50, 1.00]  # 3 fracs × 2 phases = 6 HPO rows

# Kernel restart guard: if _rows was wiped, reload from CSV to avoid losing the main sweep.
if len(_rows) < RUNS * len(FRACS) * 6:
    _path = 'results/timings_cpu.csv'
    if os.path.exists(_path):
        _rows = pd.read_csv(_path).to_dict('records')
        print(f'Reloaded {len(_rows)} rows from {_path}')
    else:
        raise RuntimeError('No CSV found — re-run section 5.2 first.')

# --- LR HPO sweep ------------------------------------------------------
for frac in HPO_FRACS:
    X_sub = X_train_raw.sample(frac=frac, random_state=42)
    y_sub = y_train.loc[X_sub.index]
    n = len(X_sub)

    X_sub_enc, _ = feature_engineering(X_sub, y=y_sub)

    pipe = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('scaler',     StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000)),
    ])
    search = GridSearchCV(
        pipe, param_grid_lr,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1', n_jobs=1, verbose=0,
    )
    t0 = time.perf_counter()
    search.fit(X_sub_enc, y_sub)
    t_this = time.perf_counter() - t0
    log_row('cpu', frac, n, 'lr_cv', run_idx=0,
            seconds=t_this,
            notes=f'{len(search.cv_results_["params"]) * 5} fits')
    print(f'lr_cv  frac={frac:.0%}  n={n:,}  elapsed={t_this/60:.1f} min')

# --- XGB HPO sweep -----------------------------------------------------
for frac in HPO_FRACS:
    X_sub = X_train_raw.sample(frac=frac, random_state=42)
    y_sub = y_train.loc[X_sub.index]
    n = len(X_sub)

    X_sub_enc, _ = feature_engineering(X_sub, y=y_sub)

    pipe = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('classifier', XGBClassifier(
            random_state=42, eval_metric='logloss',
            tree_method='hist', device='cpu',
        )),
    ])
    search = RandomizedSearchCV(
        pipe, param_grid_xgb,
        n_iter=60, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1', random_state=42, n_jobs=1, verbose=0,
    )
    t0 = time.perf_counter()
    search.fit(X_sub_enc, y_sub)
    t_this = time.perf_counter() - t0
    log_row('cpu', frac, n, 'xgb_cv', run_idx=0,
            seconds=t_this,
            notes=f'{search.n_iter * 5} fits')
    print(f'xgb_cv frac={frac:.0%}  n={n:,}  elapsed={t_this/60:.1f} min')

# Flush HPO rows to CSV.
pd.DataFrame(_rows).to_csv('results/timings_cpu.csv', index=False)
print(f'Saved results/timings_cpu.csv  ({len(_rows)} rows total)')

lr_cv  frac=25%  n=64,286  elapsed=0.6 min
lr_cv  frac=50%  n=128,572  elapsed=1.1 min
lr_cv  frac=100%  n=257,144  elapsed=2.3 min
xgb_cv frac=25%  n=64,286  elapsed=28.9 min
xgb_cv frac=50%  n=128,572  elapsed=53.4 min
xgb_cv frac=100%  n=257,144  elapsed=101.7 min
Saved results/timings_cpu.csv  (96 rows total)


### 5.4 Sanity Check

Reload the CSV from disk and verify that the row count matches the expected
**90 main + 6 HPO = 96 rows**. Print phase coverage and median seconds per (phase, fraction)
as a quick eyeball check before handing the CSV to `Analysis.ipynb`.

In [14]:
# Sanity check: reload CSV, verify row count, preview median timings.
df_cpu = pd.read_csv('results/timings_cpu.csv')

expected_main = RUNS * len(FRACS) * 6     # 3 × 5 × 6 = 90
expected_hpo  = len(HPO_FRACS) * 2        # 3 × 2 = 6
expected_rows = expected_main + expected_hpo  # 96

assert len(df_cpu) == expected_rows, (
    f'Row count mismatch: got {len(df_cpu)}, expected {expected_rows}'
)
print(f'Row count OK: {len(df_cpu)} rows')

print('\nPhase coverage:')
print(df_cpu['phase'].value_counts().sort_index().to_string())

print('\nMedian seconds per phase × fraction:')
print(
    df_cpu.groupby(['phase', 'fraction'])['seconds']
          .median()
          .unstack()
          .round(3)
          .to_string()
)

Row count OK: 96 rows

Phase coverage:
phase
fe_test        15
fe_train       15
lr_cv           3
lr_fit         15
lr_predict     15
xgb_cv          3
xgb_fit        15
xgb_predict    15

Median seconds per phase × fraction:
fraction      0.10      0.25      0.50    0.75      1.00
phase                                                   
fe_test      1.887     1.826     1.783   1.821     1.864
fe_train     0.834     2.296     3.730   7.517     9.272
lr_cv          NaN    35.520    66.776     NaN   138.055
lr_fit       0.793     1.510     4.195   4.361     6.860
lr_predict   0.119     0.104     0.105   0.083     0.119
xgb_cv         NaN  1735.517  3204.810     NaN  6102.801
xgb_fit      5.680     8.367    11.837  17.143    22.553
xgb_predict  0.753     0.693     0.706   0.978     0.703


In [15]:
# printing the total sweep time
import datetime
total_sweep_time = df_cpu['seconds'].sum()
print(f'Total measured time (all phases, all runs): {datetime.timedelta(seconds=int(total_sweep_time))}')

Total measured time (all phases, all runs): 3:14:01


In [ ]:
# Download outputs to laptop as a backup (browser save).
try:
    from google.colab import files
    for fname in ['results/best_params.json', 'results/timings_cpu.csv']:
        if os.path.exists(fname):
            files.download(fname)
            print(f'Downloading {fname}')
        else:
            print(f'WARNING: {fname} not found, skipped')
except ImportError:
    print('Not running on Colab — files are already on local disk')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>